# **Clean and Prep EASD Race Results v2**

Cleans and standardizes 5K@EASD race results by removing duplicates and translating gender values into English. Participant names are used briefly for deduplication and then discarded to maintain privacy.

In [1]:
# Import packages

# For data manipulation
import pandas as pd

# for displaying and modifying the working directory
import os as os

In [2]:
# Define custom NA values, excluding 'NA' to make sure that Country Code NA (Namibia) is not interpreted as missing data
custom_na_values = ['N/A', 'NaN', 'null', '']

In [3]:
#Load specific columns, ignore the default NA values and use the custom ones
df0 = pd.read_csv("EASD Race Results.csv", usecols=['Bib', 'Country', 'Time', 'Age Group', 'Gender'], keep_default_na=False, na_values=custom_na_values)

# Display the first 5 rows of the dataframe
df0.head()

,Bib,Country,Gender,Time,Age Group
0,2421,IN,M,00:14:45,M3539
1,2216,TD,M,00:14:45,M3034
2,3787,IN,M,00:14:45,M3539
3,4084,DZ,M,00:14:45,M4549
4,8503,DZ,M,00:14:45,M3539


In [4]:
# Display basic information about the data 
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 3737 entries, 0 to 3736
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Bib        3737 non-null   int64
 1   Country    3737 non-null   str  
 2   Gender     3722 non-null   str  
 3   Time       3737 non-null   str  
 4   Age Group  3710 non-null   str  
dtypes: int64(1), str(4)
memory usage: 146.1 KB


In [5]:
# Count number of duplicate rows based on Bib
duplicate_count = df0.duplicated(subset='Bib').sum()
print(f"Number of duplicate rows based on Bib: {duplicate_count}")

Number of duplicate rows based on Bib: 0


In [6]:
# Delete duplicate based on Bib
df0 = df0.drop_duplicates(subset='Bib')

In [7]:
# Verify that duplicates have been removed
duplicate_count_after = df0.duplicated(subset='Bib').sum()
print(f"Number of duplicate rows based on Bib after removal: {duplicate_count_after}")

Number of duplicate rows based on Bib after removal: 0


In [8]:
# Display the number of missing values in each column
df0.isnull().sum()

Bib           0
Country       0
Gender       15
Time          0
Age Group    27
dtype: int64

In [9]:
# Fill missing values in the 'Gender' column and standardize gender labels
df0['Gender'] = df0['Gender'].fillna('Unknown').replace({
    'F': 'Female',
    'M': 'Male',
    'X': 'Unknown'
})

In [10]:
# Add a column for the year
df0['Year'] = 2026

In [11]:
# Add a column for the race name
df0['Race'] = 'EASD'

In [12]:
# Rename Country to Country Code
df0 = df0.rename(columns={'Country': 'Country Code'})

In [13]:
# Change the column order
df0 = df0[['Race', 'Year', 'Country Code', 'Age Group', 'Time', 'Gender', 'Bib']]

In [14]:
# Replace null values in the 'Age Group' column with 'Unknown'
df0 = df0.fillna({'Age Group': 'Unknown'})

In [15]:
# Standardize age groups
df0['Age Group'] = df0['Age Group'].replace(
    r'^[MF](\d{1,2})(\d{2})$',
    r'\1-\2',
    regex=True
)

In [16]:
# Display count of duplicate rows
df0.duplicated().sum()

np.int64(0)

In [17]:
# Delete duplicate rows
df0 = df0.drop_duplicates()

In [18]:
# Display basic information about the data
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 3737 entries, 0 to 3736
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Race          3737 non-null   str  
 1   Year          3737 non-null   int64
 2   Country Code  3737 non-null   str  
 3   Age Group     3737 non-null   str  
 4   Time          3737 non-null   str  
 5   Gender        3737 non-null   str  
 6   Bib           3737 non-null   int64
dtypes: int64(2), str(5)
memory usage: 204.5 KB


In [19]:
# Drop the Name column
df0 = df0.drop(columns=['Bib'])

In [20]:
# If Age Group or Gender is 'Unknown', drop the row
df0 = df0[(df0['Age Group'] != 'Unknown') & (df0['Gender'] != 'Unknown')]

In [21]:
df0 = df0.sort_values(by=["Race", "Year", "Time"])

In [22]:
# Display the number of missing values in each column
df0.isnull().sum()

Race            0
Year            0
Country Code    0
Age Group       0
Time            0
Gender          0
dtype: int64

In [23]:
# Reset index
df0 = df0.reset_index(drop=True)

In [24]:
# Verify the change
df0.head()

,Race,Year,Country Code,Age Group,Time,Gender
0,EASD,2026,IN,35-39,00:14:45,Male
1,EASD,2026,TD,30-34,00:14:45,Male
2,EASD,2026,IN,35-39,00:14:45,Male
3,EASD,2026,DZ,45-49,00:14:45,Male
4,EASD,2026,DZ,35-39,00:14:45,Male


In [25]:
# Display basic information about the data 
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 3710 entries, 0 to 3709
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Race          3710 non-null   str  
 1   Year          3710 non-null   int64
 2   Country Code  3710 non-null   str  
 3   Age Group     3710 non-null   str  
 4   Time          3710 non-null   str  
 5   Gender        3710 non-null   str  
dtypes: int64(1), str(5)
memory usage: 174.0 KB


In [26]:
# Save the cleaned data to a new csv file
df0.to_csv('EASD Race Data for Visualization.csv', index=False)